# XGBoost and LSTM comparison

The reusable runner fits the models in a separate Python process so a native model-library failure cannot terminate this notebook kernel. It compares the majority baseline, logistic regression, XGBoost and LSTM across five expanding date folds on identical timestamps. Each LSTM observation uses 12 consecutive five-minute bars within one session, and scaling is fitted on training rows only.

In [1]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
from IPython.display import display
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == 'notebooks' else cwd
runner = project_root / 'python_code' / 'run_advanced_models.py'
results_dir = project_root / 'results'
metrics_file = results_dir / 'advanced_model_fold_metrics.csv'
predictions_file = results_dir / 'advanced_model_predictions.csv'

In [2]:
if metrics_file.exists() and predictions_file.exists():
    print('Using the saved model run. To recompute, run python_code/run_advanced_models.py from the project root.')
else:
    run = subprocess.run(
        [sys.executable, str(runner)],
        cwd=project_root,
        capture_output=True,
        text=True,
        check=False,
    )
    print(run.stdout)
    if run.returncode != 0:
        print(run.stderr)
        raise RuntimeError(f'Model runner exited with status {run.returncode}; notebook kernel remains separate.')

Using the saved model run. To recompute, run python_code/run_advanced_models.py from the project root.


In [3]:
fold_metrics = pd.read_csv(metrics_file)
predictions = pd.read_csv(predictions_file, parse_dates=['datetime'])
display(fold_metrics.groupby(['ETF', 'model'])[['accuracy', 'balanced_accuracy', 'f1_up', 'roc_auc']].agg(['mean', 'std']).round(4))

prediction_columns = {
    'Majority baseline': 'baseline_prediction',
    'Logistic regression': 'logistic_prediction',
    'XGBoost': 'xgboost_prediction',
    'LSTM': 'lstm_prediction',
}
pooled_rows = []
for ticker, ticker_predictions in predictions.groupby('ETF'):
    for model_name, column in prediction_columns.items():
        pooled_rows.append({
            'ETF': ticker,
            'model': model_name,
            'common_test_rows': len(ticker_predictions),
            'accuracy': (ticker_predictions['actual'] == ticker_predictions[column]).mean(),
            'up_label_rate': (ticker_predictions['actual'] == 1).mean(),
        })
display(pd.DataFrame(pooled_rows).set_index(['ETF', 'model']).round(4))

accuracy         balanced_accuracy           f1_up  \
                            mean     std              mean     std    mean   
ETF model                                                                    
QQQ LSTM                  0.5101  0.0188            0.5097  0.0193  0.5810   
    Logistic regression   0.5081  0.0122            0.5080  0.0127  0.5940   
    Majority baseline     0.5007  0.0026            0.5000  0.0000  0.6673   
    XGBoost               0.5161  0.0148            0.5160  0.0151  0.5560   
SPY LSTM                  0.5112  0.0110            0.5118  0.0115  0.5863   
    Logistic regression   0.5107  0.0099            0.5109  0.0103  0.5649   
    Majority baseline     0.4988  0.0042            0.5000  0.0000  0.6656   
    XGBoost               0.5155  0.0093            0.5156  0.0094  0.5600   

                                roc_auc          
                            std    mean     std  
ETF model                                        
QQQ LSTM                 0.0140  0.5157  0.0198  
    Logistic regression  0.0204  0.5125  0.0174  
    Majority baseline    0.0023     NaN     NaN  
    XGBoost              0.0200  0.5208  0.0122  
SPY LSTM                 0.0269  0.5188  0.0145  
    Logistic regression  0.0150  0.5160  0.0124  
    Majority baseline    0.0037     NaN     NaN  
    XGBoost              0.0222  0.5221  0.0145

common_test_rows  accuracy  up_label_rate
ETF model                                                         
QQQ Majority baseline               23419    0.5007         0.5007
    Logistic regression             23419    0.5081         0.5007
    XGBoost                         23419    0.5161         0.5007
    LSTM                            23419    0.5101         0.5007
SPY Majority baseline               23532    0.4988         0.4988
    Logistic regression             23532    0.5108         0.4988
    XGBoost                         23532    0.5155         0.4988
    LSTM                            23532    0.5112         0.4988

## Evaluation boundary

These are directional classification results using fixed initial settings, not a completed hyperparameter search. Transaction costs and trading performance belong to a separate evaluation stage. Results are written to `results/advanced_model_fold_metrics.csv` and `results/advanced_model_predictions.csv`.